# Mendeley smishing — Dataset 1 preparation

Prepares the `Smishing/smishing` subset from `v2/data/raw/SMS Mendeley.csv` for the v2 Core human-side.

**Pipeline:**
1. Filter smishing rows, normalize text, mask URLs, deduplicate.
2. Annotate each unique message with LLM-assigned `sms_fraud_subtype` via OpenRouter (cached to avoid re-calling).
3. Filter to only Core-eligible subtypes per `dataset_design_final.md §5.8`.
4. Export `scenario_family=fraud_sms_deceptive` records to `v2/data/interim/gathered/`.

**Core-eligible subtypes (§5.8) — unified policy with `smishtank_prepare.ipynb` (§5.7):**
- `account_alert`, `delivery_fee_or_service_issue`, `prize_or_contest_scam` — **include**
- `financial_or_crypto_lure`, `loan_or_credit_lure` — **exclude** (review-only per §5.8; not manually verified)
- `generic_deceptive_sms`, `unclear_other`, `lawsuit_or_settlement_lure` — exclude

**Audit fix (Apr-2026):** previous version included `financial_or_crypto_lure` and `loan_or_credit_lure`
in CORE_SUBTYPES, violating §5.7 (same policy for all SMS sources) and §5.8 (review-only, not yet verified).
Removed: −19 rows (388 → 369). `length_bin` now uses `v2/config.py` channel-level thresholds.

In [1]:
# ── 0. Setup ───────────────────────────────────────────────────────────────
from __future__ import annotations

import hashlib
import json
import os
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langdetect import detect, LangDetectException, DetectorFactory
from openai import OpenAI, APIError, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm

DetectorFactory.seed = 0  # reproducibility


def _find_v2_root() -> Path:
    c = Path(globals().get("__vsc_ipynb_file__", globals().get("__file__", "."))).resolve()
    for p in [c, *c.parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists():
            return p
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Cannot find v2/ project root")


V2_ROOT = _find_v2_root()
import sys as _sys; _sys.path.insert(0, str(V2_ROOT))
from config import length_bin as compute_length_bin  # channel-level thresholds from v2/config.py

RAW_CSV  = V2_ROOT / "data" / "raw" / "SMS Mendeley.csv"
OUT_DIR  = V2_ROOT / "data" / "interim" / "gathered"
OUT_PATH = OUT_DIR / "mendeley_smishing_prepared.jsonl"
# Annotation cache — shared with notebook 10 (reuses already-computed labels).
# New annotations are appended to the same file so the cache grows over time.
ANN_CACHE = V2_ROOT / "data" / "interim" / "annotated" / "mendeley_smishing_annotated.jsonl"

load_dotenv(V2_ROOT / ".env" if (V2_ROOT / ".env").exists() else None)
api_key = os.environ.get("OPENROUTER_API_KEY", "")
if not api_key:
    raise EnvironmentError(f"Set OPENROUTER_API_KEY in {V2_ROOT / '.env'}")

ANNOTATION_MODEL = "openai/gpt-4o-mini"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

BATCH_SIZE = 8
SLEEP_SEC  = 1.5

OUT_DIR.mkdir(parents=True, exist_ok=True)
(V2_ROOT / "data" / "interim" / "annotated").mkdir(parents=True, exist_ok=True)

print("V2_ROOT:", V2_ROOT)
print("OUT:    ", OUT_PATH)
print("Model:  ", ANNOTATION_MODEL)

V2_ROOT: /Users/askar/projects/antifraud-deepfake-detection/v2
OUT:     /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/mendeley_smishing_prepared.jsonl
Model:   openai/gpt-4o-mini


In [2]:
# ── 1. Preprocessing helpers ───────────────────────────────────────────────
URL_RE   = re.compile(r"(?i)(?:https?://\S+|www\.\S+|\b[a-z0-9][a-z0-9-]*\.[a-z]{2,}(?:/\S*)?)")
EMAIL_RE = re.compile(r"(?i)\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+?\d[\d\-\s()]{6,}\d)|(?:\b\d{5,}\b))")
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def normalize_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\uFFFD", " ")
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def mask_url(text: str) -> str:
    # First pass: catch full URLs (http://..., www.domain, bare domain)
    text = URL_RE.sub("[URL]", text)
    # Second pass: catch bare https?:// fragments left when URLs are space-broken
    text = re.sub(r"https?://\S*", "[URL]", text)
    return text


def is_english(text: str) -> bool:
    """Detect English via langdetect (seed=0 for reproducibility)."""
    if not text or len(text.strip()) < 15:
        return False
    try:
        return detect(text[:2000]) == "en"
    except LangDetectException:
        return False


def to_yn(v: bool) -> str:
    return "yes" if v else "no"


def token_length(text: str) -> int:
    return len(TOKEN_RE.findall(text))


def md5_key(text: str) -> str:
    return hashlib.md5((text or "").strip().encode("utf-8", errors="replace")).hexdigest()


# ── 2. Load and preprocess CSV ─────────────────────────────────────────────
assert RAW_CSV.exists(), f"Missing: {RAW_CSV}"
df = pd.read_csv(RAW_CSV, encoding="utf-8", keep_default_na=False)
df["_label_norm"] = df["LABEL"].astype(str).str.strip().str.lower()
smish = df[df["_label_norm"] == "smishing"].copy()

smish["text_raw"]  = smish["TEXT"].astype(str)
smish["text_norm"] = smish["text_raw"].map(normalize_text)
smish = smish[smish["text_norm"] != ""].copy()

# URL-masked text used as the output `text` field and dedup key
smish["text"] = smish["text_norm"].map(mask_url).map(normalize_text)

smish["has_url"]               = smish["text_raw"].str.contains(URL_RE)
smish["has_email"]             = smish["text_raw"].str.contains(EMAIL_RE)
smish["has_phone_or_reply_cta"] = (
    smish["text_raw"].str.contains(PHONE_RE)
    | smish["text_raw"].str.contains(r"(?i)\b(?:reply|call|text|sms|whatsapp)\b", regex=True)
)

# Language filter — keep English only
smish["is_english"] = smish["text"].map(is_english)
smish_en = smish[smish["is_english"]].copy()

# Deduplicate by masked text; keep first occurrence and count duplicates
grp = smish_en.groupby("text", sort=False)
prepared = grp.first().reset_index(drop=False)
prepared["n_duplicate_rows"] = grp.size().values

print(f"Raw rows:            {len(df)}")
print(f"Smishing rows:       {len(smish)}")
print(f"After english filter:{len(smish_en)}")
print(f"Unique masked texts: {len(prepared)}")

Raw rows:            5971
Smishing rows:       638
After english filter:637
Unique masked texts: 559


In [3]:
# ── 3. Annotation prompts and API helpers ──────────────────────────────────
SYSTEM_PROMPT = '''
You are annotating SMS fraud messages for a research dataset on LLM-generated text detection in anti-fraud systems.

Your task is NOT to summarize the message.
Your task is to assign structured labels describing the fraud-SMS subtype.

Return output as strict JSON with the following keys:

{
  "smish_subtype": "...",
  "has_link_or_url": "yes|no",
  "has_phone_or_reply_cta": "yes|no",
  "has_payment_request": "yes|no",
  "has_credential_or_verification_request": "yes|no",
  "has_urgency": "yes|no",
  "has_brand_impersonation": "yes|no",
  "confidence": "high|medium|low"
}

Allowed values for "smish_subtype":
- account_alert
- delivery_fee_or_service_issue
- prize_or_contest_scam
- financial_or_crypto_lure
- loan_or_credit_lure
- lawsuit_or_settlement_lure
- generic_deceptive_sms
- unclear_other

Definitions:
- account_alert: claims suspicious login, blocked account, profile locked, account security problem, card/account issue, verification required
- delivery_fee_or_service_issue: claims parcel problem, customs fee, failed delivery, package hold, service issue requiring action
- prize_or_contest_scam: claims winnings, contest reward, lucky draw, bonus prize, reward redemption
- financial_or_crypto_lure: invites user into crypto/investment/profit scheme, financial opportunity, trading or money-making lure
- loan_or_credit_lure: promotes deceptive credit, loan, debt, or borrowing-related bait
- lawsuit_or_settlement_lure: claims settlement money, legal compensation, lawsuit payment, class action payout
- generic_deceptive_sms: clearly deceptive or phishing-like SMS but no more specific label fits
- unclear_other: text is too noisy, ambiguous, broken, or not clearly one of the above

Annotation rules:
1. Choose the most specific subtype that fits.
2. Mark has_brand_impersonation=yes if the message clearly impersonates or invokes a specific institution, company, delivery service, bank, or brand.
3. Mark has_phone_or_reply_cta=yes if the message asks to reply, call, send a number, or otherwise respond directly.
4. Use confidence=low only when the text is strongly ambiguous or corrupted.
5. Output JSON only. No explanation.
'''

USER_TEMPLATE = '''Annotate the following SMS message for fraud subtype analysis.

Message text:
"""{sms_text}"""
'''

SUBTYPES = {
    "account_alert",
    "delivery_fee_or_service_issue",
    "prize_or_contest_scam",
    "financial_or_crypto_lure",
    "loan_or_credit_lure",
    "lawsuit_or_settlement_lure",
    "generic_deceptive_sms",
    "unclear_other",
}


def validate_ann(d: dict) -> dict:
    st = d.get("smish_subtype", "unclear_other")
    if st not in SUBTYPES:
        st = "unclear_other"

    def yn(x, default="no"):
        v = str(x).strip().lower()
        return v if v in {"yes", "no"} else default

    conf = str(d.get("confidence", "low")).strip().lower()
    if conf not in {"high", "medium", "low"}:
        conf = "low"

    return {
        "smish_subtype": st,
        "has_link_or_url": yn(d.get("has_link_or_url")),
        "has_phone_or_reply_cta_llm": yn(d.get("has_phone_or_reply_cta")),
        "has_payment_request": yn(d.get("has_payment_request")),
        "has_credential_or_verification_request": yn(d.get("has_credential_or_verification_request")),
        "has_urgency": yn(d.get("has_urgency")),
        "has_brand_impersonation": yn(d.get("has_brand_impersonation")),
        "confidence": conf,
    }


def build_messages(sms_text: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_TEMPLATE.format(sms_text=sms_text)},
    ]


@retry(
    retry=retry_if_exception_type((RateLimitError, APIError)),
    wait=wait_exponential(multiplier=2, min=2, max=45),
    stop=stop_after_attempt(6),
)
def call_json(msgs: list[dict]) -> dict:
    raw = client.chat.completions.create(
        model=ANNOTATION_MODEL,
        messages=msgs,
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=256,
    ).choices[0].message.content
    return json.loads(raw)


print("Prompt and API helpers ready.")

Prompt and API helpers ready.


In [4]:
# ── 4. Annotation loop (with cache) ───────────────────────────────────────

def load_cache(path: Path) -> dict[str, dict]:
    """Load annotation cache from JSONL file.
    Cache key = md5 of raw text (field 'text_raw' for new rows, 'text' for rows from notebook 10).
    """
    idx: dict[str, dict] = {}
    if not path.exists():
        return idx
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        # notebook 10 stores raw text in "text"; new rows store it in "text_raw"
        key_text = r.get("text_raw") or r.get("text", "")
        # notebook 10 uses "scenario_family" as subtype; new rows use "smish_subtype"
        subtype = r.get("smish_subtype") or r.get("scenario_family", "")
        if not subtype or subtype not in SUBTYPES:
            continue
        k = md5_key(key_text)
        idx[k] = {
            "smish_subtype": subtype,
            "has_link_or_url": r.get("has_link_or_url", "no"),
            "has_phone_or_reply_cta_llm": r.get("has_phone_or_reply_cta_llm") or r.get("has_phone_or_reply_cta", "no"),
            "has_payment_request": r.get("has_payment_request", "no"),
            "has_credential_or_verification_request": r.get("has_credential_or_verification_request", "no"),
            "has_urgency": r.get("has_urgency", "no"),
            "has_brand_impersonation": r.get("has_brand_impersonation", "no"),
            "confidence": r.get("annotation_confidence") or r.get("confidence", "low"),
        }
    return idx


cache = load_cache(ANN_CACHE)
print(f"Cache entries loaded: {len(cache)}")

# Build lookup: md5(text_raw) -> annotation for the deduped DataFrame
# text_raw = the raw TEXT column value before any normalization
pending_rows = [
    row for _, row in prepared.iterrows()
    if md5_key(row["text_raw"]) not in cache
]
print(f"Already annotated: {len(prepared) - len(pending_rows)} | Pending: {len(pending_rows)}")

batch_i = 0
for row in tqdm(pending_rows, desc="annotate"):
    raw_text = row["text_raw"]
    norm_text = row["text_norm"]  # normalized but NOT masked — LLM sees real URLs/phones
    try:
        ann = validate_ann(call_json(build_messages(norm_text)))
    except Exception as e:
        ann = validate_ann({})
        ann["_parse_error"] = str(e)[:240]

    ts = datetime.now(timezone.utc).isoformat()
    cache_row = {
        "text_raw": raw_text,
        "smish_subtype": ann["smish_subtype"],
        "has_link_or_url": ann["has_link_or_url"],
        "has_phone_or_reply_cta_llm": ann["has_phone_or_reply_cta_llm"],
        "has_payment_request": ann["has_payment_request"],
        "has_credential_or_verification_request": ann["has_credential_or_verification_request"],
        "has_urgency": ann["has_urgency"],
        "has_brand_impersonation": ann["has_brand_impersonation"],
        "confidence": ann["confidence"],
        "annotation_model": ANNOTATION_MODEL,
        "annotated_at": ts,
    }
    if "_parse_error" in ann:
        cache_row["_parse_error"] = ann["_parse_error"]

    # Append to shared annotation cache (same file notebook 10 uses)
    with ANN_CACHE.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(cache_row, ensure_ascii=False) + "\n")

    cache[md5_key(raw_text)] = ann

    batch_i += 1
    if batch_i >= BATCH_SIZE:
        batch_i = 0
        time.sleep(SLEEP_SEC)

print(f"\nAnnotation complete. Cache size: {len(cache)}")

Cache entries loaded: 561
Already annotated: 559 | Pending: 0


annotate: 0it [00:00, ?it/s]


Annotation complete. Cache size: 561


In [5]:
# ── 5. Filter and build final output ──────────────────────────────────────
# Subtypes to include per dataset_design_final.md §5.8 — unified policy with smishtank (§5.7).
# financial_or_crypto_lure and loan_or_credit_lure are review-only (not yet manually verified);
# excluded here to keep the same inclusion rule as smishtank_prepare.ipynb.
CORE_SUBTYPES = {
    "account_alert",
    "delivery_fee_or_service_issue",
    "prize_or_contest_scam",
}

out_rows: list[dict] = []
skipped_no_ann = 0
skipped_excluded = 0

for _, row in prepared.iterrows():
    ann = cache.get(md5_key(row["text_raw"]))
    if ann is None:
        skipped_no_ann += 1
        continue

    subtype = ann["smish_subtype"]
    if subtype not in CORE_SUBTYPES:
        skipped_excluded += 1
        continue

    text = row["text"]  # URL-masked output text
    char_len  = len(text)
    tok_len   = token_length(text)

    out_rows.append({
        "text": text,
        "label": 0,
        "label_str": "human",
        "fraudness": "fraud",
        "channel": "sms",
        "scenario_family": "fraud_sms_deceptive",
        "sms_fraud_subtype": subtype,
        "source_family": "mendeley_sms_phishing",
        "dataset_source": "SMS Mendeley.csv",
        "has_url": to_yn(bool(row["has_url"])),
        "has_phone_or_reply_cta": to_yn(bool(row["has_phone_or_reply_cta"])),
        "has_email": to_yn(bool(row["has_email"])),
        "char_length": char_len,
        "token_length": tok_len,
        "length_bin": compute_length_bin(tok_len, "sms"),  # from v2/config.py
        "time_band": "legacy",
        "origin_model": "human",
        "split": "tbd",
        "lang_filter_method": "langdetect_v1",
        "n_duplicate_rows": int(row["n_duplicate_rows"]),
        "annotation_confidence": ann["confidence"],
        "annotation_model": ANNOTATION_MODEL,
    })

# Write output
with OUT_PATH.open("w", encoding="utf-8") as fh:
    for r in out_rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Total unique masked texts:  {len(prepared)}")
print(f"Skipped — no annotation:    {skipped_no_ann}")
print(f"Skipped — excluded subtype: {skipped_excluded}")
print(f"Written to output:          {len(out_rows)}")
print(f"Output file: {OUT_PATH}")
print()
print("sms_fraud_subtype distribution:")
import collections
for subtype, cnt in sorted(collections.Counter(r["sms_fraud_subtype"] for r in out_rows).items(), key=lambda x: -x[1]):
    print(f"  {subtype:40s} {cnt}")

Total unique masked texts:  559
Skipped — no annotation:    0
Skipped — excluded subtype: 190
Written to output:          369
Output file: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/mendeley_smishing_prepared.jsonl

sms_fraud_subtype distribution:
  prize_or_contest_scam                    288
  account_alert                            72
  delivery_fee_or_service_issue            9


In [6]:
# ── 6. Sanity checks ───────────────────────────────────────────────────────
assert OUT_PATH.exists(), "Output file was not created"
check = pd.read_json(OUT_PATH, lines=True)

assert check["sms_fraud_subtype"].isna().sum() == 0, "Missing subtypes"
assert set(check["sms_fraud_subtype"]).issubset(CORE_SUBTYPES), "Unexpected subtype in output"
assert (check["scenario_family"] == "fraud_sms_deceptive").all()
assert (check["label"] == 0).all()
assert (check["channel"] == "sms").all()
assert set(check["length_bin"]).issubset({"short", "medium", "long"}), "unexpected length_bin values"
assert check["text"].str.contains(r"https?://|www\.", regex=True).sum() == 0, \
    "Some output texts still contain unmasked URLs"

print(f"Rows:              {len(check)}")
print(f"Missing subtypes:  {int(check['sms_fraud_subtype'].isna().sum())}")
print(f"Columns:           {list(check.columns)}")
print()
print("sms_fraud_subtype distribution:")
print(check["sms_fraud_subtype"].value_counts())
print()
print("length_bin distribution:")
print(check["length_bin"].value_counts())
print()
print("Sample per subtype (first 2):")
for sub in sorted(check["sms_fraud_subtype"].unique()):
    print(f"\n== {sub} ==")
    for txt in check.loc[check["sms_fraud_subtype"] == sub, "text"].head(2):
        print(" ", txt[:200])

print()
print("Files in gathered/:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)

Rows:              369
Missing subtypes:  0
Columns:           ['text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family', 'sms_fraud_subtype', 'source_family', 'dataset_source', 'has_url', 'has_phone_or_reply_cta', 'has_email', 'char_length', 'token_length', 'length_bin', 'time_band', 'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows', 'annotation_confidence', 'annotation_model']

sms_fraud_subtype distribution:
sms_fraud_subtype
prize_or_contest_scam            288
account_alert                     72
delivery_fee_or_service_issue      9
Name: count, dtype: int64

length_bin distribution:
length_bin
medium    346
short      21
long        2
Name: count, dtype: int64

Sample per subtype (first 2):

== account_alert ==
  BankOfAmerica Alert 137943. Please follow [URL] re-activate
  Apple ID: [BUXCX7GBVwWCcOD Final Notification Your Apple 1D is due to expire today. Prevent this by confirming your Apple ID at [URL] Apple Inc

== delivery_fee_or_service_issue 